In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from shapely.wkt import loads

In [2]:
dfCrime = pd.read_csv('/Users/austincoffelt/Documents/Rent-Walk-Local/Calls_for_Service_2018_20250419.csv')
temp = pd.read_csv('/Users/austincoffelt/Documents/Rent-Walk-Local/Calls_for_Service_2019_20250428.csv')
tem2 = pd.read_csv('/Users/austincoffelt/Documents/Rent-Walk-Local/Calls_for_Service_2017_20250430.csv')
dfCrime = pd.concat([dfCrime, temp, tem2])
print(dfCrime.shape)
dfCrime.head()

(1392347, 23)


,NOPD_Item,Type_,TypeText,Priority,InitialType,InitialTypeText,InitialPriority,MapX,MapY,TimeCreate,...,Disposition,DispositionText,SelfInitiated,Beat,BLOCK_ADDRESS,Zip,PoliceDistrict,Location,Type,TimeArrival
0,A1964518,67C,THEFT FROM EXTERIOR,1A,62C,SIMPLE BURGLARY VEHICLE,1G,0,0,01/17/2018 04:10:29 PM,...,GOA,GONE ON ARRIVAL,N,7L03,147XX Chef Menteur Hwy,NaN,0,NaN,NaN,NaN
1,A1213518,21,COMPLAINT OTHER,1J,21,COMPLAINT OTHER,1J,0,0,01/11/2018 12:35:07 AM,...,NAT,Necessary Action Taken,Y,8D01,Bourbon St & Iberville St,NaN,0,NaN,NaN,NaN
2,A0838218,37,AGGRAVATED ASSAULT,1C,103,DISTURBANCE (OTHER),2A,0,0,01/08/2018 01:24:10 AM,...,RTF,REPORT TO FOLLOW,N,8D06,007XX Bourbon St,NaN,0,NaN,NaN,NaN
3,B0948918,21P,LOST PROPERTY,0G,21P,LOST PROPERTY,0G,37369000,3513814,02/09/2018 01:18:23 AM,...,RTF,REPORT TO FOLLOW,N,8D05,004XX Bourbon St,70130.0,0,"(0.0, 0.0)",NaN,NaN
4,A0609918,24,MEDICAL,2B,29,DEATH,1B,37369000,3513814,01/06/2018 01:15:19 AM,...,NAT,Necessary Action Taken,N,2L04,016XX Pine St,70118.0,2,"(0.0, 0.0)",NaN,NaN


In [3]:
sorted(list(dfCrime['TypeText'].unique()))

['ABANDONED BOAT',
 'ABANDONED VEHICLE',
 'AGGRAVATED ARSON',
 'AGGRAVATED ASSAULT',
 'AGGRAVATED ASSAULT DOMESTIC',
 'AGGRAVATED BATTERY',
 'AGGRAVATED BATTERY BY CUTTING',
 'AGGRAVATED BATTERY BY KNIFE',
 'AGGRAVATED BATTERY BY SHOOTING',
 'AGGRAVATED BATTERY DOMESTIC',
 'AGGRAVATED BURGLARY',
 'AGGRAVATED BURGLARY DOMESTIC',
 'AGGRAVATED CRIMINAL DAMAGE',
 'AGGRAVATED ESCAPE',
 'AGGRAVATED KIDNAPPING',
 'AGGRAVATED RAPE',
 'AGGRAVATED RAPE MALE VICTIM',
 'AGGRAVATED RAPE UNFOUNDED BY SPECIAL VICTIMS OR CHILD ABUSE',
 'AIRPLANE CRASH',
 'AREA CHECK',
 'ARMED ROBBERY',
 'ARMED ROBBERY WITH GUN',
 'ARMED ROBBERY WITH KNIFE',
 'ASSET SEIZURE',
 'ATTACHMENT',
 'AUTO ACCIDENT',
 'AUTO ACCIDENT CITY VEHICLE',
 'AUTO ACCIDENT FATALITY',
 'AUTO ACCIDENT POLICE VEHICLE',
 'AUTO ACCIDENT WITH INJURY',
 'AUTO THEFT',
 'BICYCLE THEFT',
 'BLIGHTED PROPERTY',
 'BOMB SCARE',
 'BURGLAR ALARM, LOCAL',
 'BURGLAR ALARM, SILENT',
 'BURGLAR ALARM, SILENT, FAR NO RESPONSE',
 'BUSINESS BURGLARY',
 'BUSINES

In [4]:
violentCrimes = ['AGGRAVATED ASSAULT', 'AGGRAVATED ASSAULT DOMESTIC', 'AGGRAVATED RAPE', 'AGGRAVATED RAPE MALE VICTIM', 
                 'AGGRAVATED RAPE UNFOUNDED BY SPECIAL VICTIMS OR CHILD ABUSE', 'ARMED ROBBERY', 'ARMED ROBBERY WITH GUN', 
                 'ARMED ROBBERY WITH KNIFE', 'HOMICIDE', 'HOMICIDE BY CUTTING', 'HOMICIDE BY SHOOTING', 'HOMICIDE DOMESTIC',
                'SIMPLE RAPE', 'SIMPLE RAPE MALE VICTIM', 'SIMPLE RAPE UNFOUNDED BY SPECIAL VICTIMS OR CHILD ABUSE']
propertyCrimes = ['BUSINESS BURGLARY', 'CARJACKING', 'CARJACKING- NO WEAPON',  'RESIDENCE BURGLARY', 'SAFE BURGLARY', 'SIMPLE BURGLARY',
                  'SIMPLE BURGLARY DOMESTIC', 'SIMPLE BURGLARY VEHICLE', 'THEFT', 'THEFT FROM EXTERIOR OF VEHICLE', 'AGGRAVATED BURGLARY',
                  'AGGRAVATED BURGLARY DOMESTIC', 'AUTO THEFT', 'BICYCLE THEFT']
both = violentCrimes + propertyCrimes
dfCrime = dfCrime[dfCrime['TypeText'].isin(both)]
print(dfCrime.shape)
dfCrime.head()

(69309, 23)


,NOPD_Item,Type_,TypeText,Priority,InitialType,InitialTypeText,InitialPriority,MapX,MapY,TimeCreate,...,Disposition,DispositionText,SelfInitiated,Beat,BLOCK_ADDRESS,Zip,PoliceDistrict,Location,Type,TimeArrival
2,A0838218,37,AGGRAVATED ASSAULT,1C,103,DISTURBANCE (OTHER),2A,0,0,01/08/2018 01:24:10 AM,...,RTF,REPORT TO FOLLOW,N,8D06,007XX Bourbon St,NaN,0,NaN,NaN,NaN
70,B1357518,37,AGGRAVATED ASSAULT,1E,37,AGGRAVATED ASSAULT,1E,3694198,559742,02/11/2018 11:54:41 PM,...,RTF,REPORT TO FOLLOW,N,7A05,Downman Rd & Hayne Blvd,70126.0,3,"(30.03270567, -90.0269583)",NaN,NaN
126,A1607318,37,AGGRAVATED ASSAULT,1A,37,AGGRAVATED ASSAULT,1E,3670226,535972,01/14/2018 08:49:04 AM,...,UNF,UNFOUNDED,N,1L04,039XX Tulane Ave,70119.0,1,"(29.96807474, -90.10351781)",NaN,NaN
155,A1620818,67,THEFT,0A,67,THEFT,1G,3681879,531499,01/14/2018 12:21:05 PM,...,GOA,GONE ON ARRIVAL,N,8C02,003XX Royal St,70130.0,8,"(29.95542795, -90.06687159)",NaN,NaN
200,A0367218,62R,RESIDENCE BURGLARY,2A,62,SIMPLE BURGLARY,2A,3705177,512629,01/03/2018 11:36:51 PM,...,RTF,REPORT TO FOLLOW,N,4D05,057XX Tullis Dr,70131.0,4,"(29.90281221, -89.99400291)",NaN,NaN


In [6]:
dfCrime = dfCrime[['TypeText', 'Location']]
dfCrime = dfCrime[dfCrime['Location'] != '(0.0, 0.0)']
dfCrime = dfCrime[dfCrime['Location'].isna() == False]
dfCrime.head()

,TypeText,Location
70,AGGRAVATED ASSAULT,"(30.03270567, -90.0269583)"
126,AGGRAVATED ASSAULT,"(29.96807474, -90.10351781)"
155,THEFT,"(29.95542795, -90.06687159)"
200,RESIDENCE BURGLARY,"(29.90281221, -89.99400291)"
219,ARMED ROBBERY WITH GUN,"(29.94109943, -90.07916897)"


In [7]:
dfCrime.dtypes

TypeText    object
Location    object
dtype: object

In [8]:
# Extract lat and lon using regex (handles parentheses and commas)
dfCrime[['lat', 'lon']] = dfCrime['Location'].str.extract(r'\(([^,]+),\s*([^)]+)\)').astype(float)

# Create Point geometry
dfCrime['geometry'] = dfCrime.apply(lambda row: Point(row['lon'], row['lat']), axis=1)

dfCrime = dfCrime[['TypeText', 'Location', 'geometry']]

In [9]:
# Read shapefile (automatically loads .shp, .shx, .dbf)
gdf = gpd.read_file("/Users/austincoffelt/Documents/Rent-Walk-Local/cb_2018_22_bg_500k/cb_2018_22_bg_500k.shp")

# View the first 5 rows
print(gdf.shape)
gdf.head()

(3463, 11)


,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,AFFGEOID,GEOID,NAME,LSAD,ALAND,AWATER,geometry
0,22,051,021801,1,1500000US220510218011,220510218011,1,BG,759364,0,"POLYGON ((-90.187 30.012, -90.182 30.012, -90...."
1,22,051,027703,2,1500000US220510277032,220510277032,2,BG,371159,0,"POLYGON ((-90.118 29.848, -90.117 29.848, -90...."
2,22,009,030100,1,1500000US220090301001,220090301001,1,BG,562491448,34183405,"POLYGON ((-92.053 31.162, -92.048 31.168, -92...."
3,22,001,961000,2,1500000US220019610002,220019610002,2,BG,590930,0,"POLYGON ((-92.372 30.209, -92.371 30.209, -92...."
4,22,079,012100,1,1500000US220790121001,220790121001,1,BG,1179564,0,"POLYGON ((-92.463 31.291, -92.463 31.292, -92...."


In [10]:
# make sure points match the crs of gdf
gdfPoints = gpd.GeoDataFrame(
    dfCrime, 
    geometry='geometry',
    crs="EPSG:4269"  # WGS84 coordinate reference system
)
gdfPoints.head()

,TypeText,Location,geometry
70,AGGRAVATED ASSAULT,"(30.03270567, -90.0269583)",POINT (-90.027 30.033)
126,AGGRAVATED ASSAULT,"(29.96807474, -90.10351781)",POINT (-90.104 29.968)
155,THEFT,"(29.95542795, -90.06687159)",POINT (-90.067 29.955)
200,RESIDENCE BURGLARY,"(29.90281221, -89.99400291)",POINT (-89.994 29.903)
219,ARMED ROBBERY WITH GUN,"(29.94109943, -90.07916897)",POINT (-90.079 29.941)


In [11]:
# Basic spatial join (points within polygons)
result = gpd.sjoin(
    gdfPoints, 
    gdf,
    how="left",       # 'left', 'right', or 'inner'
    predicate="within" # Spatial relationship
)
print(result.shape)
result.head()

(67148, 14)


,TypeText,Location,geometry,index_right,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,AFFGEOID,GEOID,NAME,LSAD,ALAND,AWATER
70,AGGRAVATED ASSAULT,"(30.03270567, -90.0269583)",POINT (-90.027 30.033),2516.0,22,071,001720,4,1500000US220710017204,220710017204,4,BG,1986504.0,233905.0
126,AGGRAVATED ASSAULT,"(29.96807474, -90.10351781)",POINT (-90.104 29.968),2008.0,22,071,007101,2,1500000US220710071012,220710071012,2,BG,535661.0,0.0
155,THEFT,"(29.95542795, -90.06687159)",POINT (-90.067 29.955),780.0,22,071,013500,1,1500000US220710135001,220710135001,1,BG,369821.0,186999.0
200,RESIDENCE BURGLARY,"(29.90281221, -89.99400291)",POINT (-89.994 29.903),2040.0,22,071,000617,2,1500000US220710006172,220710006172,2,BG,447795.0,137853.0
219,ARMED ROBBERY WITH GUN,"(29.94109943, -90.07916897)",POINT (-90.079 29.941),913.0,22,071,013900,2,1500000US220710139002,220710139002,2,BG,255692.0,0.0


In [12]:
crime = result['AFFGEOID'].value_counts() / 3
crime

AFFGEOID
1500000US220710134003    981.666667
1500000US220710135001    491.666667
1500000US220710134002    416.333333
1500000US220710134001    390.000000
1500000US220710017511    307.333333
                            ...    
1500000US220510258002      2.333333
1500000US220510249002      1.333333
1500000US220710011003      1.000000
1500000US220510250021      0.333333
1500000US220870302045      0.333333
Name: count, Length: 500, dtype: float64

In [13]:
# export as a csv
crime.to_csv('/Users/austincoffelt/Documents/RW Data/NO_All_Crime_BG.csv')

In [14]:
result2 = result[result['TypeText'].isin(violentCrimes)]
crime = result2['AFFGEOID'].value_counts() / 3
crime

AFFGEOID
1500000US220710134003    60.666667
1500000US220710017511    42.333333
1500000US220710017204    31.666667
1500000US220710135001    31.000000
1500000US220710134002    27.333333
                           ...    
1500000US220710126001     0.333333
1500000US220710091002     0.333333
1500000US220710116001     0.333333
1500000US220710096002     0.333333
1500000US220710076063     0.333333
Name: count, Length: 475, dtype: float64

In [15]:
# export as a csv
crime.to_csv('/Users/austincoffelt/Documents/RW Data/NO_Violent_Crime_BG.csv')

In [16]:
result3 = result[result['TypeText'].isin(propertyCrimes)]
crime = result3['AFFGEOID'].value_counts() / 3
crime

AFFGEOID
1500000US220710134003    921.000000
1500000US220710135001    460.666667
1500000US220710134002    389.000000
1500000US220710134001    376.333333
1500000US220710135003    281.333333
                            ...    
1500000US220710007011      5.000000
1500000US220710070001      5.000000
1500000US220510258002      2.333333
1500000US220710011003      0.666667
1500000US220510249002      0.666667
Name: count, Length: 498, dtype: float64

In [17]:
# export as a csv
crime.to_csv('/Users/austincoffelt/Documents/RW Data/NO_Property_Crime_BG.csv')

In [18]:
result4 = result[result['TypeText'].isin(['AUTO THEFT'])]
crime = result4['AFFGEOID'].value_counts() / 3
crime

AFFGEOID
1500000US220710134003    102.000000
1500000US220710134002     64.333333
1500000US220710017511     63.333333
1500000US220710134001     58.333333
1500000US220710135001     56.000000
                            ...    
1500000US220710056013      1.000000
1500000US220710070001      1.000000
1500000US220710121022      0.666667
1500000US220710076041      0.666667
1500000US220510249002      0.333333
Name: count, Length: 496, dtype: float64

In [19]:
# export as a csv
crime.to_csv('/Users/austincoffelt/Documents/RW Data/NO_MV_Crime_BG.csv')